# Set up for dataset and model

Package installation, loading, and dataloaders. There's also a resnet18 model defined.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from tqdm import tqdm

from torchvision import datasets, transforms

use_cuda = True
device = torch.device("cuda" if use_cuda else "cpu")
batch_size = 64

np.random.seed(42)
torch.manual_seed(42)


## Dataloaders
test_dataset = datasets.CIFAR10('cifar10_data/', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor()]
))

test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [2]:
def tp_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return .5 * (x + delta) * (1 - ind1) * (1 - ind2) + x * ind2


def tp_smoothed_relu(x, delta=1.):
    ind1 = (x < -1. * delta).float()
    ind2 = (x > delta).float()
    return (x + delta) ** 2 / (4 * delta) * (1 - ind1) * (1 - ind2) + x * ind2


class Normalize(nn.Module):
    def __init__(self, mu, std):
        super(Normalize, self).__init__()
        self.mu, self.std = mu, std

    def forward(self, x):
        return (x - self.mu) / self.std


class IdentityLayer(nn.Module):
    def forward(self, inputs):
        return inputs


class PreActBlock(nn.Module):
    '''Pre-activation version of the BasicBlock.'''
    expansion = 1

    def __init__(self, in_planes, planes, bn, learnable_bn, stride=1, activation='relu'):
        super(PreActBlock, self).__init__()
        self.collect_preact = True
        self.activation = activation
        self.avg_preacts = []
        self.bn1 = nn.BatchNorm2d(in_planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not learnable_bn)
        self.bn2 = nn.BatchNorm2d(planes, affine=learnable_bn) if bn else IdentityLayer()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not learnable_bn)

        if stride != 1 or in_planes != self.expansion * planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, self.expansion * planes, kernel_size=1, stride=stride, bias=not learnable_bn)
            )

    def act_function(self, preact):
        if self.activation == 'relu':
            act = F.relu(preact)
        elif self.activation[:6] == '3prelu':
            act = tp_relu(preact, delta=float(self.activation.split('relu')[1]))
        elif self.activation[:8] == '3psmooth':
            act = tp_smoothed_relu(preact, delta=float(self.activation.split('smooth')[1]))
        else:
            assert self.activation[:8] == 'softplus'
            beta = int(self.activation.split('softplus')[1])
            act = F.softplus(preact, beta=beta)
        return act

    def forward(self, x):
        out = self.act_function(self.bn1(x))
        shortcut = self.shortcut(out) if hasattr(self, 'shortcut') else x  # Important: using out instead of x
        out = self.conv1(out)
        out = self.conv2(self.act_function(self.bn2(out)))
        out += shortcut
        return out


class PreActResNet(nn.Module):
    def __init__(self, block, num_blocks, n_cls, cuda=True, half_prec=False,
                 activation='relu', fts_before_bn=False, normal='none'):
        super(PreActResNet, self).__init__()
        self.bn = True
        self.learnable_bn = True  # doesn't matter if self.bn=False
        self.in_planes = 64
        self.avg_preact = None
        self.activation = activation
        self.fts_before_bn = fts_before_bn
        if normal == 'cifar10':
            self.mu = torch.tensor((0.4914, 0.4822, 0.4465)).view(1, 3, 1, 1)
            self.std = torch.tensor((0.2471, 0.2435, 0.2616)).view(1, 3, 1, 1)
        else:
            self.mu = torch.tensor((0.0, 0.0, 0.0)).view(1, 3, 1, 1)
            self.std = torch.tensor((1.0, 1.0, 1.0)).view(1, 3, 1, 1)
            print('no input normalization')
        if cuda:
            self.mu = self.mu.cuda()
            self.std = self.std.cuda()
        if half_prec:
            self.mu = self.mu.half()
            self.std = self.std.half()

        self.normalize = Normalize(self.mu, self.std)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not self.learnable_bn)
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.bn = nn.BatchNorm2d(512 * block.expansion)
        self.linear = nn.Linear(512 * block.expansion, n_cls)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, self.bn, self.learnable_bn, stride, self.activation))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x, return_features=False):
        for layer in [*self.layer1, *self.layer2, *self.layer3, *self.layer4]:
            layer.avg_preacts = []

        out = self.normalize(x)
        out = self.conv1(out)
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        if return_features and self.fts_before_bn:
            return out.view(out.size(0), -1)
        out = F.relu(self.bn(out))
        if return_features:
            return out.view(out.size(0), -1)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        out = self.linear(out)

        return out


def PreActResNet18(n_cls, cuda=True, half_prec=False, activation='relu', fts_before_bn=False,
                   normal='none'):
    return PreActResNet(PreActBlock, [2, 2, 2, 2], n_cls=n_cls, cuda=cuda, half_prec=half_prec,
                        activation=activation, fts_before_bn=fts_before_bn, normal=normal)


# intialize the model
model = PreActResNet18(10, cuda=True, activation='softplus1').to(device)
model.eval()

no input normalization


PreActResNet(
  (normalize): Normalize()
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (layer1): Sequential(
    (0): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
    (1): PreActBlock(
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    )
  )
  (layer2): Sequential(
    (0): PreActBloc

# Implement the Attacks

Both attacks are untargeted, so each step *ascends* the cross-entropy loss of the
**true** label, then projects back onto the eps-ball around the original `x` and
onto the image domain `[0, 1]`.

* **Linf**: the step is `eps_step * sign(grad)` and the projection is a coordinate-wise
  clamp of the perturbation to `[-eps, eps]`.
* **L2**: the gradient is normalised to unit L2 norm per example,
  `g / (||g||_2 + 1e-10)`, and the projection rescales the perturbation by
  `eps / max(||delta||_2, eps)` (a no-op when already inside the ball).

In [3]:
DELTA = 1e-10   # guards the divide-by-zero in the L2 gradient normalisation


def pgd_linf_untargeted(model, x, labels, k, eps, eps_step, random_start=False):
    """Untargeted Linf PGD: k steps of FGSM, each projected back onto B_inf(x, eps).

    Projection onto the Linf ball is just a coordinate-wise clamp of the
    perturbation to [-eps, eps]; we additionally clamp to the image domain [0,1].
    """
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    x = x.detach()
    adv_x = x.clone().detach()
    if random_start:
        adv_x = adv_x + torch.empty_like(adv_x).uniform_(-eps, eps)
        adv_x = torch.clamp(adv_x, 0.0, 1.0).detach()
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, labels)
        loss.backward()
        with torch.no_grad():
            # untargeted => ASCEND the loss of the true label
            adv_x = adv_x + eps_step * adv_x.grad.sign()
            # project onto the eps-ball around the ORIGINAL x, then onto [0,1]
            delta = torch.clamp(adv_x - x, min=-eps, max=eps)
            adv_x = torch.clamp(x + delta, 0.0, 1.0)
        adv_x = adv_x.detach()
    return adv_x


def pgd_l2_untargeted(model, x, labels, k, eps, eps_step, random_start=False):
    """Untargeted L2 PGD.

    Step:        x^(t+1) = x^(t) + eta * g^(t) / (||g^(t)||_2 + delta)
    Projection:  x_proj  = x + eps * (x^(t+1) - x) / max(||x^(t+1) - x||_2, eps)

    Both the gradient normalisation and the projection are computed per example
    in the batch, so the flattened norms are taken over the C*H*W dimensions.
    """
    model.eval()
    ce_loss = torch.nn.CrossEntropyLoss()
    x = x.detach()
    adv_x = x.clone().detach()
    batch_size = x.size()[0]
    if random_start:
        d = torch.randn_like(adv_x)
        d = d / (d.view(batch_size, -1).norm(p=2, dim=1).view(-1, 1, 1, 1) + DELTA)
        r = torch.rand(batch_size, 1, 1, 1, device=x.device) ** (1.0 / d[0].numel())
        adv_x = torch.clamp(adv_x + eps * r * d, 0.0, 1.0).detach()
    for _ in range(k):
        adv_x.requires_grad_(True)
        model.zero_grad()
        output = model(adv_x)
        loss = ce_loss(output, labels)
        loss.backward()
        grad = adv_x.grad.data
        with torch.no_grad():
            # normalise the gradient to unit L2 norm per example
            grad_norms = grad.view(batch_size, -1).norm(p=2, dim=1) + DELTA
            grad = grad / grad_norms.view(batch_size, 1, 1, 1)
            adv_x = adv_x + eps_step * grad

            # project the perturbation back onto the L2 ball of radius eps
            delta = adv_x - x
            delta_norms = delta.view(batch_size, -1).norm(p=2, dim=1)
            factor = eps / torch.clamp(delta_norms, min=eps)   # == eps / max(||delta||, eps)
            delta = delta * factor.view(batch_size, 1, 1, 1)

            adv_x = torch.clamp(x + delta, 0.0, 1.0)
        adv_x = adv_x.detach()
    return adv_x

# Evaluate Single and Multi-Norm Robust Accuracy

In this section, we evaluate the model on the Linf and L2 attacks as well as union accuracy.

In [4]:
K = 10   # number of PGD steps


def load_model(path):
    sd = torch.load(path, map_location=device, weights_only=False)
    sd = {k.replace('module.', ''): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    assert not missing and not unexpected
    model.eval()
    return model


def test_model_on_single_attack(model, attack='pgd_linf', eps=0.1, k=K):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        if attack == 'pgd_linf':
            x_adv = pgd_linf_untargeted(model, x_batch, y_batch, k, eps, eps / 4)
        elif attack == 'pgd_l2':
            x_adv = pgd_l2_untargeted(model, x_batch, y_batch, k, eps, eps / 4)
        else:
            x_adv = x_batch          # no attack -> standard accuracy

        with torch.no_grad():
            pred = torch.max(model(x_adv), dim=1)[1]
        tot_acc += (pred == y_batch).sum().item()
        tot_test += x_batch.size(0)

    acc = tot_acc / tot_test
    label = 'Standard accuracy' if attack not in ('pgd_linf', 'pgd_l2') else 'Robust accuracy'
    print('%s %.5lf' % (label, acc), f'on {attack} attack with eps = {eps}')
    return acc

## Standard (clean) accuracy

In [5]:
for name in ['Linf', 'L2', 'RAMP']:
    load_model(f'models/pretr_{name}.pth')
    print(f'=== {name} ===')
    test_model_on_single_attack(model, attack='none')

=== Linf ===


Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating:   1%|          | 1/157 [00:00<00:40,  3.87it/s]

Evaluating:   4%|▍         | 7/157 [00:00<00:06, 23.79it/s]

Evaluating:  10%|▉         | 15/157 [00:00<00:03, 41.23it/s]

...

Evaluating:  86%|████████▌ | 135/157 [00:01<00:00, 72.33it/s]

Evaluating:  91%|█████████ | 143/157 [00:02<00:00, 73.17it/s]

Evaluating:  96%|█████████▌| 151/157 [00:02<00:00, 72.20it/s]

Evaluating: 100%|██████████| 157/157 [00:02<00:00, 69.65it/s]

Standard accuracy 0.81190 on none attack with eps = 0.1


## Single-Norm Robust Accuracy

In [6]:
# Evaluate on Linf attack with different models with eps = 8/255
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_linf', eps=8. / 255.)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_linf', eps=8. / 255.)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_linf', eps=8. / 255.)

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating:   1%|          | 1/157 [00:00<00:55,  2.81it/s]

Evaluating:   1%|▏         | 2/157 [00:00<00:49,  3.15it/s]

Evaluating:   2%|▏         | 3/157 [00:00<00:49,  3.11it/s]

Evaluating:   3%|▎         | 4/157 [00:01<00:46,  3.27it/s]

...

Evaluating:  99%|█████████▊| 155/157 [00:38<00:00,  4.20it/s]

Evaluating:  99%|█████████▉| 156/157 [00:38<00:00,  4.18it/s]

Evaluating: 100%|██████████| 157/157 [00:38<00:00,  4.10it/s]

Robust accuracy 0.49750 on pgd_linf attack with eps = 0.03137254901960784


0.4975

In [7]:
# Evaluate on L2 attack with different models with eps = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
model.eval()
test_model_on_single_attack(model, attack='pgd_l2', eps=0.75)

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating:   1%|          | 1/157 [00:00<00:38,  4.09it/s]

Evaluating:   1%|▏         | 2/157 [00:00<00:37,  4.10it/s]

Evaluating:   2%|▏         | 3/157 [00:00<00:37,  4.13it/s]

Evaluating:   3%|▎         | 4/157 [00:00<00:36,  4.15it/s]

...

Evaluating:  99%|█████████▊| 155/157 [00:37<00:00,  4.15it/s]

Evaluating:  99%|█████████▉| 156/157 [00:37<00:00,  4.18it/s]

Evaluating: 100%|██████████| 157/157 [00:37<00:00,  4.16it/s]

Robust accuracy 0.59650 on pgd_l2 attack with eps = 0.75


0.5965

## Multi-Norm Robust Accuracy

Union accuracy over `Delta = B_2(x, eps_2) U B_inf(x, eps_inf)`: a point counts as
correct only if the model classifies it correctly under **both** adversaries. A single
point that either attack breaks is counted as a failure.

In [8]:
def test_model_on_multi_attacks(model, eps_linf=8. / 255., eps_l2=0.75, k=K):
    model.eval()
    tot_test, tot_acc = 0.0, 0.0
    for batch_idx, (x_batch, y_batch) in tqdm(enumerate(test_loader), total=len(test_loader), desc="Evaluating"):
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        x_adv_linf = pgd_linf_untargeted(model, x_batch, y_batch, k, eps_linf, eps_linf / 4)
        x_adv_l2 = pgd_l2_untargeted(model, x_batch, y_batch, k, eps_l2, eps_l2 / 4)

        ## calculate union accuracy: correct only if both attacks are correct

        with torch.no_grad():
            out = model(x_adv_linf)
            pred_linf = torch.max(out, dim=1)[1]
            out = model(x_adv_l2)
            pred_l2 = torch.max(out, dim=1)[1]

        correct = (pred_linf == y_batch) & (pred_l2 == y_batch)
        tot_acc += correct.sum().item()
        tot_test += x_batch.size(0)

    print('Robust accuracy %.5lf' % (tot_acc / tot_test), f'on multi attacks')
    return tot_acc / tot_test

In [9]:
# Evaluate on multi-norm attacks with different models with eps_linf = 8./255, eps_l2 = 0.75
model.load_state_dict(torch.load('models/pretr_Linf.pth'))
model.eval()
test_model_on_multi_attacks(model)

model.load_state_dict(torch.load('models/pretr_L2.pth'))
model.eval()
test_model_on_multi_attacks(model)

model.load_state_dict(torch.load('models/pretr_RAMP.pth'))
model.eval()
test_model_on_multi_attacks(model)

Evaluating:   0%|          | 0/157 [00:00<?, ?it/s]

Evaluating:   1%|          | 1/157 [00:00<01:13,  2.14it/s]

Evaluating:   1%|▏         | 2/157 [00:00<01:13,  2.12it/s]

Evaluating:   2%|▏         | 3/157 [00:01<01:12,  2.12it/s]

Evaluating:   3%|▎         | 4/157 [00:01<01:12,  2.12it/s]

...

Evaluating:  99%|█████████▉| 156/157 [01:14<00:00,  2.01it/s]

Evaluating: 100%|██████████| 157/157 [01:14<00:00,  2.54it/s]

Evaluating: 100%|██████████| 157/157 [01:14<00:00,  2.10it/s]

Robust accuracy 0.49740 on multi attacks


0.4974

## Results

See `WRITEUP.md` for the full tables, the convergence check over the number of PGD
steps, and the discussion of the accuracy differences between the three models.